# Skenario 2 — Cross-Language Information Retrieval (CLIR)
## Dense Retrieval dengan mmBERT-small (Baseline vs Fine-Tuned)

**Alur kerja notebook ini:**
1. Instalasi dependensi & setup environment
2. Import library dan inisialisasi PyTerrier
3. Definisi kelas `FaissRetriever` untuk dense retrieval
4. Load & preprocessing data (dokumen, kueri, qrels)
5. Eksperimen evaluasi: Baseline vs Fine-Tuned
6. Analisis hasil per-kueri dan penyimpanan laporan

## 1. Instalasi Dependensi & Setup Environment

Menginstall semua library yang dibutuhkan, meng-clone repository kode, dan menginisialisasi PyTerrier.
- **faiss-cpu** digunakan sebagai fallback aman karena `faiss-gpu` sering tidak tersedia di Kaggle.
- **sentence-transformers** untuk encoding teks menjadi vektor dense.
- **pyarabic** untuk normalisasi teks Arab.

In [6]:
import os
import sys
import subprocess
import torch

# --- 1a. Cek Ketersediaan GPU ---
if torch.cuda.is_available():
    print(f"✅ GPU Terdeteksi: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU tidak terdeteksi. Encoding akan berjalan di CPU (lebih lambat).")

# --- 1b. Install Library ---
print("\nMenginstall dependensi...")
libs = ["sentence-transformers", "python-terrier", "pyarabic", "faiss-cpu", "pandas", "tashaphyne"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + libs)
print("✅ Instalasi selesai.")

# --- 1c. Clone Repository ---
REPO_PATH = './skripsi-clir-code'
SRC_PATH  = os.path.join(REPO_PATH, 'src')

if os.path.exists(REPO_PATH):
    print("Repository sudah ada. Melakukan pull untuk update terbaru...")
    os.system(f"cd {REPO_PATH} && git pull")
else:
    print("Cloning repository...")
    os.system("git clone https://github.com/syifaurrr/skripsi-clir-code.git")

# Tambahkan folder src ke sys.path agar modul custom bisa di-import
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)
print(f"✅ Path '{SRC_PATH}' ditambahkan ke sys.path.")

✅ GPU Terdeteksi: Tesla T4

Menginstall dependensi...
✅ Instalasi selesai.
Repository sudah ada. Melakukan pull untuk update terbaru...
Already up to date.
✅ Path './skripsi-clir-code/src' ditambahkan ke sys.path.


## 2. Import Library & Inisialisasi PyTerrier

Mengimpor semua modul yang dibutuhkan, termasuk modul preprocessing Arab dari repository.
PyTerrier diinisialisasi sekali saja di awal karena memerlukan JVM (Java Virtual Machine).

In [7]:
import re
import numpy as np
import pandas as pd
import faiss
import pyterrier as pt
import pyarabic.araby as araby
from sentence_transformers import SentenceTransformer
from IPython.display import display

# Import modul preprocessing dari repository
try:
    from arabic_preprocessing import preprocess_pipeline
    print("✅ Modul arabic_preprocessing berhasil di-import.")
except ImportError as e:
    print(f"⚠️ Modul arabic_preprocessing tidak ditemukan: {e}")
    print("Pastikan struktur folder di repository benar: skripsi-clir-code/src/arabic_preprocessing.py")

# Inisialisasi PyTerrier (hanya dilakukan sekali)
if not pt.started():
    print("Menginisialisasi PyTerrier...")
    pt.init()
    print("✅ PyTerrier siap.")

✅ Modul arabic_preprocessing berhasil di-import.


/tmp/ipykernel_55/1443752540.py:19: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


## 3. Definisi Kelas `FaissRetriever`

Kelas ini membungkus FAISS sebagai `pt.Transformer` agar kompatibel dengan pipeline PyTerrier.

**Cara kerja:**
- Saat inisialisasi, dokumen di-encode menjadi vektor dense menggunakan SentenceTransformer.
- Vektor dokumen disimpan dalam FAISS `IndexFlatIP` (Inner Product ≈ Cosine Similarity setelah normalisasi L2).
- Saat `transform()` dipanggil, vektor kueri di-encode lalu dicari Top-1000 dokumen terdekat di index.

In [8]:
class FaissRetriever(pt.Transformer):
    """
    Dense retriever berbasis FAISS + SentenceTransformer.
    Kompatibel dengan pipeline pt.Experiment (PyTerrier).
    """
    def __init__(self, model_name, doc_df=None, batch_size=32):
        self.model      = SentenceTransformer(model_name)
        self.index      = None
        self.doc_map    = {}
        self.batch_size = batch_size

        # Patch token_type_ids yang tidak didukung beberapa model
        tokenizer = getattr(self.model, 'tokenizer', None)
        if tokenizer and 'token_type_ids' in tokenizer.model_input_names:
            tokenizer.model_input_names.remove('token_type_ids')

        # Langsung buat index jika dokumen diberikan saat konstruksi
        if doc_df is not None:
            self._index_docs(doc_df)

    def _index_docs(self, doc_df):
        """Encode semua dokumen dan simpan ke index FAISS."""
        print(f"Encoding {len(doc_df)} dokumen (mungkin memakan waktu beberapa menit)...")
        docs    = doc_df['text_clean'].tolist()
        docnos  = doc_df['docno'].tolist()

        embeddings = self.model.encode(
            docs, batch_size=self.batch_size,
            show_progress_bar=True, convert_to_numpy=True
        )
        faiss.normalize_L2(embeddings)  # Wajib untuk Cosine Similarity via Inner Product

        dim        = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)

        # Peta index FAISS (integer) -> docno asli (string)
        self.doc_map = {i: docno for i, docno in enumerate(docnos)}
        print("✅ Indexing selesai.")

    def transform(self, queries):
        """Encode kueri dan kembalikan Top-1000 hasil retrieval."""
        q_texts      = queries['query'].tolist()
        q_embeddings = self.model.encode(
            q_texts, batch_size=self.batch_size,
            show_progress_bar=False, convert_to_numpy=True
        )
        faiss.normalize_L2(q_embeddings)

        k = 1000
        scores, ids = self.index.search(q_embeddings, k)

        results = []
        for i, qid in enumerate(queries['qid']):
            for j in range(k):
                idx = ids[i][j]
                if idx != -1:
                    results.append({
                        'qid'   : str(qid),
                        'docno' : self.doc_map[idx],
                        'score' : scores[i][j],
                        'rank'  : j + 1
                    })
        return pd.DataFrame(results)

## 4. Load & Preprocessing Data

Memuat tiga berkas data utama:
- **Dokumen** (`fathul_muin.csv`): Korpus teks berbahasa Arab yang akan diindeks.
- **Kueri** (`queries_indo.csv`): Pertanyaan dalam Bahasa Indonesia.
- **Qrels** (`qrels.csv`): Pasangan relevansi kueri-dokumen sebagai ground truth evaluasi.

Preprocessing dokumen Arab meliputi normalisasi karakter (harakat, tatweel, alef, ya/waw).
ID di semua dataframe distandarkan ke format string bersih untuk menghindari mismatch saat evaluasi.

In [9]:
# --- 4a. Fungsi Normalisasi Teks Arab ---
def normalize_arabic(text):
    """Membersihkan dan menormalisasi teks Arab."""
    if not isinstance(text, str):
        return ""
    text = araby.strip_tashkeel(text)   # Hapus harakat/tanda baca
    text = araby.strip_tatweel(text)    # Hapus tatweel (pemanjang)
    text = araby.normalize_alef(text)   # Normalisasi variasi huruf Alef
    text = re.sub(r'[ى]',  'ي', text)  # Alef maqsurah -> Ya
    text = re.sub(r'[ؤ]',  'و', text)  # Waw hamza -> Waw
    text = re.sub(r'[ئ]',  'ي', text)  # Ya hamza  -> Ya
    return text

# --- 4b. Konfigurasi Path Data ---
DATA_PATH       = './skripsi-clir-code/data'
raw_docs_path   = os.path.join(DATA_PATH, 'raw/fathul_muin.csv')
queries_path    = os.path.join(DATA_PATH, 'queries/queries_indo.csv')
qrels_path      = os.path.join(DATA_PATH, 'queries/qrels.csv')

# --- 4c. Load Dokumen ---
df_docs = pd.read_csv(raw_docs_path)
text_col = 'text' if 'text' in df_docs.columns else 'text_arabic'
df_docs['docno']      = df_docs['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
# df_docs['text_clean'] = df_docs[text_col].astype(str).apply(normalize_arabic)
# tanpa normalisasi
df_docs['text_clean'] = df_docs[text_col].astype(str)

# --- 4d. Load Kueri ---
df_queries = pd.read_csv(queries_path)
df_queries['qid'] = df_queries['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# --- 4e. Load Qrels ---
qrels = pd.read_csv(qrels_path)
qrels['qid']   = qrels['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['docno'] = qrels['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['label'] = qrels['label'].astype(int)

# --- Verifikasi format ID ---
print("Verifikasi format ID:")
print(f"  Sampel qid (kueri)  : {df_queries['qid'].iloc[0]!r}")
print(f"  Sampel qid (qrels)  : {qrels['qid'].iloc[0]!r}")
print(f"  Sampel docno (docs) : {df_docs['docno'].iloc[0]!r}")
print(f"  Sampel docno (qrels): {qrels['docno'].iloc[0]!r}")
print(f"\nJumlah dokumen : {len(df_docs)}")
print(f"Jumlah kueri   : {len(df_queries)}")
print(f"Jumlah qrels   : {len(qrels)}")

Verifikasi format ID:
  Sampel qid (kueri)  : '1'
  Sampel qid (qrels)  : '1'
  Sampel docno (docs) : 'Page_V01P031'
  Sampel docno (qrels): 'Page_V01P038'

Jumlah dokumen : 639
Jumlah kueri   : 153
Jumlah qrels   : 153


## 5. Eksperimen Evaluasi: Baseline vs Fine-Tuned

Membandingkan dua model:
1. **Baseline** — `jhu-clsp/mmBERT-small` tanpa fine-tuning.
2. **Fine-Tuned** — Model yang sudah dilatih ulang dengan dataset domain (JH-POLO).

Eksperimen dijalankan dengan `perquery=True` agar dapat menganalisis performa per kueri.
Metrik yang digunakan:
- **Reciprocal Rank (RR)** — Kebalikan dari peringkat dokumen relevan pertama.
- **Success@K** (K = 10, 20, 50, 100) — Apakah dokumen relevan ditemukan dalam Top-K?

In [10]:
EVAL_METRICS = ["recip_rank", "success_10", "success_20", "success_50", "success_100"]

# --- 5a. Inisialisasi Model Baseline ---
print("Memuat Model Baseline (tanpa fine-tuning)...")
BASE_MODEL_NAME = 'jhu-clsp/mmBERT-small'
retriever_base  = FaissRetriever(
    model_name=BASE_MODEL_NAME,
    doc_df=df_docs[['docno', 'text_clean']],
    batch_size=32
)

# --- 5b. Inisialisasi Model Fine-Tuned ---
# ⚠️ Sesuaikan path ini dengan lokasi model fine-tuned Anda di Kaggle Datasets
print("\nMemuat Model Fine-Tuned (JH-POLO)...")
# FINETUNED_MODEL_PATH = '/kaggle/input/datasets/syifaur/mmbert-small-bs8-ep6-lr5e-05-wr0-1-seq128'
# NORM
FINETUNED_MODEL_PATH = '/kaggle/input/datasets/syifaur/mmbert-small-bs8-ep6-lr5e-05-wr0-1-seq128-norm'
retriever_finetuned  = FaissRetriever(
    model_name=FINETUNED_MODEL_PATH,
    doc_df=df_docs[['docno', 'text_clean']],
    batch_size=32
)

# --- 5c. Jalankan Eksperimen PyTerrier ---
print("\nMenjalankan pt.Experiment (perquery=True)...")
hasil_eval_perq = pt.Experiment(
    [retriever_base, retriever_finetuned],
    df_queries,
    qrels,
    eval_metrics=EVAL_METRICS,
    names=["Baseline (mmBERT-small)", "mmBERT-small Fine-Tuned (JH-POLO)"],
    validate="ignore",
    perquery=True
)

# --- 5d. Reshape: format panjang -> format lebar (metrik jadi kolom) ---
hasil_pivot = hasil_eval_perq.pivot_table(
    index=['name', 'qid'],
    columns='measure',
    values='value'
).reset_index()

# Merge metadata kueri (tipe & teks query) untuk analisis lebih lanjut
hasil_pivot['qid']   = hasil_pivot['qid'].astype(str)
df_queries['qid']    = df_queries['qid'].astype(str)

hasil_dengan_tipe = pd.merge(
    hasil_pivot,
    df_queries[['qid', 'query_type', 'query']],
    on='qid', how='left'
)
print("✅ Eksperimen selesai.")

No sentence-transformers model found with name jhu-clsp/mmBERT-small. Creating a new one with mean pooling.


Memuat Model Baseline (tanpa fine-tuning)...


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: jhu-clsp/mmBERT-small
Key               | Status     |  | 
------------------+------------+--+-
decoder.weight    | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cac

Encoding 639 dokumen (mungkin memakan waktu beberapa menit)...


...

✅ Indexing selesai.

Memuat Model Fine-Tuned (JH-POLO)...


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Encoding 639 dokumen (mungkin memakan waktu beberapa menit)...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

✅ Indexing selesai.

Menjalankan pt.Experiment (perquery=True)...
✅ Eksperimen selesai.


## 6. Analisis & Penyimpanan Hasil

Hasil evaluasi ditampilkan dalam tiga level:
1. **Overall** — Rata-rata metrik seluruh kueri untuk tiap model.
2. **Per tipe kueri** — Rata-rata metrik dikelompokkan berdasarkan kategori kueri.
3. **Detail per kueri** — Informasi rank dokumen relevan ditemukan dan label Hit/Miss@K untuk tiap kueri.

Semua tabel disimpan sebagai CSV untuk keperluan penulisan skripsi.

In [11]:
# Helper: konversi kolom success ke persen dan bulatkan recip_rank
def format_metrics(df, groupby_cols):
    result = df.groupby(groupby_cols)[EVAL_METRICS].mean().reset_index()
    for k in [10, 20, 50, 100]:
        result[f'success_{k} (%)'] = (result[f'success_{k}'] * 100).round(2)
    result['recip_rank'] = result['recip_rank'].round(4)
    return result

DISPLAY_COLS_BASE = ['name', 'recip_rank', 'success_10 (%)', 'success_20 (%)', 'success_50 (%)', 'success_100 (%)']

# --- 6a. Hasil Keseluruhan (Overall) ---
print("=" * 60)
print("🎯 HASIL KESELURUHAN (OVERALL)")
print("=" * 60)
overall_metrics = format_metrics(hasil_dengan_tipe, ['name'])
display(overall_metrics[DISPLAY_COLS_BASE])

# --- 6b. Hasil Per Tipe Kueri ---
print("\n" + "=" * 60)
print("📊 HASIL BERDASARKAN TIPE KUERI")
print("=" * 60)
per_type_metrics = format_metrics(hasil_dengan_tipe, ['name', 'query_type'])
display(per_type_metrics[['name', 'query_type'] + DISPLAY_COLS_BASE[1:]])

# --- 6c. Analisis Detail Per Kueri (Hit/Miss@K) ---
print("\n" + "=" * 60)
print("🔍 ANALISIS DETAIL PER KUERI")
print("=" * 60)

detail_kueri = hasil_dengan_tipe.copy()

# Konversi Reciprocal Rank -> estimasi peringkat dokumen relevan
detail_kueri['rank_ditemukan'] = detail_kueri['recip_rank'].apply(
    lambda x: int(round(1 / x)) if x > 0 else "Tidak Ketemu"
)
# Label visual Hit/Miss untuk tiap threshold K
for k in [10, 20, 50, 100]:
    detail_kueri[f'Hit@{k}'] = detail_kueri[f'success_{k}'].apply(
        lambda x: '✅ Hit' if x == 1.0 else '❌ Miss'
    )

# Urutkan: Model -> Tipe Kueri -> Nomor Kueri (numerik)
detail_kueri['qid_int'] = pd.to_numeric(detail_kueri['qid'], errors='coerce').fillna(0).astype(int)
detail_kueri = detail_kueri.sort_values(by=['name', 'query_type', 'qid_int']).drop(columns='qid_int')

KOLOM_TAMPIL = ['name', 'qid', 'query_type', 'query', 'rank_ditemukan',
                'Hit@10', 'Hit@20', 'Hit@50', 'Hit@100']
display(detail_kueri[KOLOM_TAMPIL].head(20))

# --- 6d. Simpan Semua Laporan ke CSV ---
detail_kueri[KOLOM_TAMPIL].to_csv('skenario2_mmbert-small_detail_per_kueri.csv',        index=False)
per_type_metrics.to_csv(          'skenario2_mmbert-small_evaluasi_per_tipe_kueri.csv', index=False)
overall_metrics.to_csv(           'skenario2_mmbert-small_evaluasi_overall.csv',        index=False)

print("\n📑 Seluruh laporan berhasil disimpan:")
print("  - skenario2_mmbert-small_detail_per_kueri.csv")
print("  - skenario2_mmbert-small_evaluasi_per_tipe_kueri.csv")
print("  - skenario2_mmbert-small_evaluasi_overall.csv")

🎯 HASIL KESELURUHAN (OVERALL)


,name,recip_rank,success_10 (%),success_20 (%),success_50 (%),success_100 (%)
0,Baseline (mmBERT-small),0.0131,1.31,6.54,17.65,28.76
1,mmBERT-small Fine-Tuned (JH-POLO),0.2318,45.10,52.94,68.63,78.43



📊 HASIL BERDASARKAN TIPE KUERI


,name,query_type,recip_rank,success_10 (%),success_20 (%),success_50 (%),success_100 (%)
0,Baseline (mmBERT-small),1,0.0135,1.96,7.84,15.69,27.45
1,Baseline (mmBERT-small),2,0.0145,1.96,5.88,17.65,31.37
2,Baseline (mmBERT-small),3,0.0112,0.00,5.88,19.61,27.45
3,mmBERT-small Fine-Tuned (JH-POLO),1,0.2287,39.22,47.06,56.86,68.63
4,mmBERT-small Fine-Tuned (JH-POLO),2,0.2350,47.06,60.78,74.51,80.39
5,mmBERT-small Fine-Tuned (JH-POLO),3,0.2317,49.02,50.98,74.51,86.27



🔍 ANALISIS DETAIL PER KUERI


,name,qid,query_type,query,rank_ditemukan,Hit@10,Hit@20,Hit@50,Hit@100
0,Baseline (mmBERT-small),1,1,Kewajiban orang tua untuk memerintahkan sholat...,28,❌ Miss,❌ Miss,✅ Hit,✅ Hit
87,Baseline (mmBERT-small),4,1,Kesunnahan membaca basmalah dalam wudhu,78,❌ Miss,❌ Miss,❌ Miss,✅ Hit
120,Baseline (mmBERT-small),7,1,Alasan-alasan yang membatalkan wudhu,198,❌ Miss,❌ Miss,❌ Miss,❌ Miss
1,Baseline (mmBERT-small),10,1,Syarat sah sholat yang kedua,170,❌ Miss,❌ Miss,❌ Miss,❌ Miss
34,Baseline (mmBERT-small),13,1,Tata cara mensucikan najis (air seni) yang tel...,450,❌ Miss,❌ Miss,❌ Miss,❌ Miss
61,Baseline (mmBERT-small),16,1,Kewajiban mengingatkan seseorang yang pakaiann...,340,❌ Miss,❌ Miss,❌ Miss,❌ Miss
64,Baseline (mmBERT-small),19,1,Kesunahan membaca doa qunut di dalam sholat subuh,120,❌ Miss,❌ Miss,❌ Miss,❌ Miss
68,Baseline (mmBERT-small),22,1,Solusi bagi makmum ketika lupa membaca surah a...,18,❌ Miss,✅ Hit,✅ Hit,✅ Hit
71,Baseline (mmBERT-small),25,1,Hukum melakukan sunnah ab'ad yang terlupakan s...,478,❌ Miss,❌ Miss,❌ Miss,❌ Miss
74,Baseline (mmBERT-small),28,1,Kesunnahan adzan pada selain sholat fardhu,458,❌ Miss,❌ Miss,❌ Miss,❌ Miss



📑 Seluruh laporan berhasil disimpan:
  - skenario2_mmbert-small_detail_per_kueri.csv
  - skenario2_mmbert-small_evaluasi_per_tipe_kueri.csv
  - skenario2_mmbert-small_evaluasi_overall.csv
